# This notebook describes the workflow for extracting fitbit data from the SQL tables into CSV files

## Set up your environment
You will want to set your Cloud compute profile to have: 8 CPUs, 30GB RAM, 1000GB Disk (note that this is at the bottom) and the Compute type should be a Dataproc Cluster You'll note that there's a handy link nearby that you can use to monitor your usage. We like to use the Yarn dashboard.

You will want to pick 4 Workers and 8 Preemptible workers with the following config 8 CPUs, 30 GB RAM and 1000GB Disk

Go ahead and restart your environment!

In [ ]:
#import necessary libraries
import pandas as pd
import os
import glob
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import gcsfs
import time
## Get the workspace project info so that we can use this when pulling data out of BigQuery
bucket_ = os.environ.get('WORKSPACE_BUCKET')

#paths that are hardcoded to save files generated
file_paths_values = pd.read_csv('filepaths_values.csv')
file_paths_values = file_paths_values[file_paths_values['notebook']=='data_grab'][['notebook','key_','value_']]
paths_dict = file_paths_values.set_index("key_")["value_"]

In [ ]:
#folder names to save data
folder_ehr_save = paths_dict.get("ehr_save")
folder_no_ehr_save = paths_dict.get("no_ehr_save")

## Create person chunks
V8 has about 60,000 individuals. Grabbing all their data together is not feasible. So we processed data in chunks of 1000 from two mains sets(people with EHR data and people without EHR data) The following code creates fixed splits that will be used throughout while processing the data

### Subjects with fitbit data with EHR data as well

In [ ]:
# This query grabs the person_ids for people with fitbit and ehr data from domain "person" table and was generated for All of Us Controlled Tier Dataset v8
dataset_97932049_person_sql = """
    SELECT
        person.person_id,
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_fitbit = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )"""

dataset_97932049_person_df = pd.read_gbq(
    dataset_97932049_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_97932049_person_df.head(5)

In [ ]:
#we should have 36,614 people for fitbit+ehr set
len(set(dataset_97932049_person_df['person_id']))

In [ ]:
#chunk out in groups of 1000
pids = sorted(list(set(dataset_97932049_person_df['person_id'])))
c = 1
for i in range(0,len(pids),1000):
    if i+1000<len(pids):
        file_path = f"{bucket_}/{folder_ehr_save}/chunk_{c}.txt"
        fs = gcsfs.GCSFileSystem()
        with fs.open(file_path, "w") as gcs_file:
            for pid in pids[i:i+1000]:
                gcs_file.write(f"{pid}\n")  

        print(f"File successfully saved to {file_path}")
    else:
        file_path = f"{bucket_}/{folder_ehr_save}/chunk_{c}.txt"
        fs = gcsfs.GCSFileSystem()
        with fs.open(file_path, "w") as gcs_file:
            for pid in pids[i:len(pids)]:
                gcs_file.write(f"{pid}\n")  

        print(f"File successfully saved to {file_path}")
    c+=1

In [ ]:
#use this to load each set and check if everything was saved correctly
def load_person_ids_from_file(file_path: str) -> list:
    """Load person IDs from a text file."""
    fs = gcsfs.GCSFileSystem()
    with fs.open(file_path, 'r') as file:
        person_ids = [int(line.strip()) for line in file.readlines()]
    return person_ids
pids_r  = []
for jj in range(1,38):
    pids_r = pids_r + load_person_ids_from_file(f'{bucket_}/{folder_ehr_save}/chunk_{jj}.txt')

In [ ]:
len(set(pids_r).intersection(set(pids)))

### Subjects with fitbit data but no EHR data
Repeat the above same process for people without ehr and chunk them too. Use 'has_ehr_data = 0' in sql query. You should have 22,404 people

## Grab Fitbit data

### Steps data

In [ ]:
#load a chunk to process
def load_person_ids_from_file(file_path: str) -> list:
    """Load person IDs from a text file."""
    fs = gcsfs.GCSFileSystem()
    with fs.open(file_path, 'r') as file:
        person_ids = [int(line.strip()) for line in file.readlines()]
    return person_ids
no = 1 #change depending on chunk
pids_to_write = load_person_ids_from_file(f'{bucket_}/{folder_ehr_save}/chunk_{no}.txt')

This query grabs minute by miute step array (aggregates second level data at minute level if present), 
along with available timestamps and orders by minutes of day.  
[current versions only have minute data but since heart rate data changed with version update adding for future use]  
  
It also inserts -1 in the array if a timestamp is missing so it is truly a 1440 array representing correct step counts for corresponding minute of day.  


In [ ]:
person_ids_str = ', '.join(map(str, pids_to_write))

dataset_hr_mapped_level_sql = """
WITH minute_level_data AS (
    SELECT
        person_id,
        src_id,
        DATETIME_TRUNC(datetime, MINUTE) as minute_timestamp,
        CAST(DATETIME_TRUNC(datetime, MINUTE) AS DATE) AS date,
        CAST(SUM(steps) AS INT) as step_count
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.steps_intraday`
    WHERE
        person_id IN (""" + person_ids_str + """)
    GROUP BY
        person_id,
        src_id,
        minute_timestamp,
        date
),
-- Generate complete minute range for each person-date combination
date_ranges AS (
    SELECT DISTINCT
        person_id,
        src_id,
        date
    FROM minute_level_data
),
-- Create all 1440 minutes for each day (00:00 to 23:59)
complete_minutes AS (
    SELECT
        dr.person_id,
        dr.src_id,
        dr.date,
        DATETIME_ADD(
            DATETIME(dr.date), 
            INTERVAL minute_offset MINUTE
        ) as minute_timestamp
    FROM date_ranges dr
    CROSS JOIN UNNEST(GENERATE_ARRAY(0, 1439)) AS minute_offset
),
-- Join with actual data and fill missing values with -1
complete_data AS (
    SELECT
        cm.person_id,
        cm.src_id,
        cm.date,
        cm.minute_timestamp,
        COALESCE(mld.step_count, -1) as stepcount
    FROM complete_minutes cm
    LEFT JOIN minute_level_data mld
        ON cm.person_id = mld.person_id
        AND cm.src_id = mld.src_id
        AND cm.date = mld.date
        AND cm.minute_timestamp = mld.minute_timestamp
),
minute_aggregated AS (
SELECT
    person_id,
    src_id,
    date,
    
    -- Complete arrays (always 1440 entries per day)
    ARRAY_TO_STRING(ARRAY_AGG(CAST(stepcount AS STRING) ORDER BY minute_timestamp), ',') AS steps_array_str,
    
    -- Summary statistics (only for non-zero values)
    COUNTIF(stepcount > 0) as non_zero_step_count,
    
FROM complete_data
GROUP BY
    person_id,
    src_id,
    date
ORDER BY
    person_id,
    src_id,
    date
)

   
SELECT
    ma.person_id,
    ma.src_id,
    ma.date,
    ma.steps_array_str,
    ma.non_zero_step_count
FROM minute_aggregated ma
ORDER BY
    ma.person_id,
    ma.src_id,
    ma.date
"""
processed_df = pd.read_gbq(
    dataset_hr_mapped_level_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")




In [ ]:
# #save as needed
processed_df[["person_id", "date", "non_zero_step_count", "steps_array_str"]].to_csv(f'{bucket_}/{folder_ehr_save}/chunk_{no}/fitbit_intraday_updated_steps_chunk_{no}.tsv',sep='\t',index = False)


### Heart rate data

In [ ]:
#load a chunk to process
def load_person_ids_from_file(file_path: str) -> list:
    """Load person IDs from a text file."""
    fs = gcsfs.GCSFileSystem()
    with fs.open(file_path, 'r') as file:
        person_ids = [int(line.strip()) for line in file.readlines()]
    return person_ids
no = 1 #change depending on chunk
pids_to_write = load_person_ids_from_file(f'{bucket_}/{folder_ehr_save}/chunk_{no}.txt')

This query grabs minute by miute heart rate array (aggregates second level data at minute level if present), 
along with available timestamps and orders by minutes of day.  
It also inserts 0 in the array if a timestamp is missing so it is truly a 1440 array representing correct heart rate value for corresponding minute of day.  
It also grabs a second level heart rate array and its timestamps for people that have it avalaible (i.e src_id = careevolution)


In [ ]:
person_ids_str = ', '.join(map(str, pids_to_write))

dataset_hr_mapped_level_sql = """
WITH minute_level_data AS (
    SELECT
        person_id,
        src_id,
        DATETIME_TRUNC(datetime, MINUTE) as minute_timestamp,
        CAST(DATETIME_TRUNC(datetime, MINUTE) AS DATE) AS date,
        CAST(AVG(heart_rate_value) AS INT) as avg_heart_rate
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.heart_rate_minute_level`
    WHERE
        person_id IN (""" + person_ids_str + """)
    GROUP BY
        person_id,
        src_id,
        minute_timestamp,
        date
),
-- Generate complete minute range for each person-date combination
date_ranges AS (
    SELECT DISTINCT
        person_id,
        src_id,
        date
    FROM minute_level_data
),
-- Create all 1440 minutes for each day (00:00 to 23:59)
complete_minutes AS (
    SELECT
        dr.person_id,
        dr.src_id,
        dr.date,
        DATETIME_ADD(
            DATETIME(dr.date), 
            INTERVAL minute_offset MINUTE
        ) as minute_timestamp
    FROM date_ranges dr
    CROSS JOIN UNNEST(GENERATE_ARRAY(0, 1439)) AS minute_offset
),
-- Join with actual data and fill missing values with 0
complete_data AS (
    SELECT
        cm.person_id,
        cm.src_id,
        cm.date,
        cm.minute_timestamp,
        COALESCE(mld.avg_heart_rate, 0) as heart_rate
    FROM complete_minutes cm
    LEFT JOIN minute_level_data mld
        ON cm.person_id = mld.person_id
        AND cm.src_id = mld.src_id
        AND cm.date = mld.date
        AND cm.minute_timestamp = mld.minute_timestamp
),
minute_aggregated AS (
SELECT
    person_id,
    src_id,
    date,
    
    -- Complete arrays (always 1440 entries per day)
    ARRAY_TO_STRING(ARRAY_AGG(CAST(heart_rate AS STRING) ORDER BY minute_timestamp), ',') AS heart_rate_array_str,
    
    -- Summary statistics (only for non-zero values)
    COUNTIF(heart_rate > 0) as non_zero_count,
    
FROM complete_data
GROUP BY
    person_id,
    src_id,
    date
ORDER BY
    person_id,
    src_id,
    date
),

raw_second_level AS (
SELECT
    person_id,
    src_id,
    CAST(datetime AS DATE) AS date,
    ARRAY_TO_STRING(ARRAY_AGG(CAST(datetime AS STRING) ORDER BY datetime), ',') AS sec_timestamps_str, -- Raw datetime values as an array
    ARRAY_TO_STRING(ARRAY_AGG(CAST(heart_rate_value AS STRING) ORDER BY datetime), ',') AS sec_heart_rate_array_str, -- Corresponding heart rate values
    COUNT(heart_rate_value) as nos_available
FROM
    `""" + os.environ["WORKSPACE_CDR"] + """.heart_rate_minute_level`
WHERE
    person_id IN (""" + person_ids_str + """) 
GROUP BY
    person_id,
    src_id,
    date
)    
SELECT
    ma.person_id,
    ma.src_id,
    ma.date,
    ma.heart_rate_array_str,
    ma.non_zero_count,
    rsl.sec_timestamps_str,
    rsl.sec_heart_rate_array_str,
    COALESCE(rsl.nos_available, ma.non_zero_count) as raw_data_count
    
FROM minute_aggregated ma
LEFT JOIN raw_second_level rsl
    ON ma.person_id = rsl.person_id
    AND ma.src_id = rsl.src_id  
    AND ma.date = rsl.date
    
    AND (
             (rsl.nos_available>1440) OR (rsl.src_id = 'careevolution')
        )
    
    
ORDER BY
    ma.person_id,
    ma.src_id,
    ma.date
"""
processed_df = pd.read_gbq(
    dataset_hr_mapped_level_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")




In [ ]:
#save as needed
processed_df.to_csv(f'{bucket_}/{folder_ehr_save}/chunk_{no}/fitbit_heart_rate_level_second_chunk_{no}.tsv',sep='\t',index = False)

### Sleep data

In [ ]:
#load a chunk to process
def load_person_ids_from_file(file_path: str) -> list:
    """Load person IDs from a text file."""
    fs = gcsfs.GCSFileSystem()
    with fs.open(file_path, 'r') as file:
        person_ids = [int(line.strip()) for line in file.readlines()]
    return person_ids
no = 1 #change depending on chunk
pids_to_write = load_person_ids_from_file(f'{bucket_}/{folder_ehr_save}/chunk_{no}.txt')

In [ ]:
#This query grabs data on teh sleep stages fro every night of sleep recorded

person_ids_str = ', '.join(map(str, pids_to_write))

dataset_97932049_fitbit_sleep_level_sql = """
    SELECT
        * 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.sleep_level` sleep_level   
    WHERE
        PERSON_ID IN (""" + person_ids_str + """)"""
dataset_97932049_fitbit_sleep_level_df = pd.read_gbq(
    dataset_97932049_fitbit_sleep_level_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_97932049_fitbit_sleep_level_df.head(5)

In [ ]:
#save as needed
dataset_97932049_fitbit_sleep_level_df.to_csv(f'{bucket_}/{folder_ehr_save}/chunk_{no}/fitbit_sleep_data_chunk_{no}.tsv',sep='\t',index = False)

### Activity summary data

In [ ]:
#load a chunk to process
def load_person_ids_from_file(file_path: str) -> list:
    """Load person IDs from a text file."""
    fs = gcsfs.GCSFileSystem()
    with fs.open(file_path, 'r') as file:
        person_ids = [int(line.strip()) for line in file.readlines()]
    return person_ids
no = 1 #change depending on chunk
pids_to_write = load_person_ids_from_file(f'{bucket_}/{folder_ehr_save}/chunk_{no}.txt')

In [ ]:
#This query grabs data on the daily activity stats (moderate active minutes, sedentary minutes, etc)

person_ids_str = ', '.join(map(str, pids_to_write))

dataset_97932049_fitbit_activity_summary_sql = """
    SELECT
        * 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.activity_summary` activity_summary 
    WHERE
        PERSON_ID IN (""" + person_ids_str + """)"""
dataset_97932049_fitbit_activity_summary_df = pd.read_gbq(
    dataset_97932049_fitbit_activity_summary_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

# dataset_97932049_fitbit_activity_summary_df.head(5)
dataset_97932049_fitbit_activity_summary_df.head(5)

In [ ]:
#save as needed
dataset_97932049_fitbit_activity_summary_df.to_csv(f'{bucket_}/{folder_ehr_save}/chunk_{no}/fitbit_activity_summary_chunk_{no}.tsv',sep='\t',index = False)

# Sleep Quality control 
Cleans and aggregate sleep segment data into basic summary statistics
### Rules to QC the data
Remove duplicate entries that are identical in [‘person_id’, ‘level’, ‘start_datetime’, ‘duration_in_min’] but differ only in ‘is_main_sleep  
Removing entries (sleep stage record) that have a duration of more than 1440 minutes  
Account for records moving into the next day around midnight time  
Account for overlap in times between classic and/or stages - give priority to “stages” data, Within each, truncate earlier at next start  
Flag invalid days if total duration > 1440  


### QC constants and functions

In [ ]:


# Priority rules used during overlap resolution
# Lower number = higher priority (1 is highest)
PRIORITY_TABLE: Dict[str, int] = {
    "deep": 1,
    "wake": 1,
    "light": 1,
    "rem": 1,  
    "asleep": 2,
    "restless": 2,
    "awake": 2,
    "unknown": 3,  # considered, but lower priority
    # anything not listed -> 4 (ignored)
}

PRIORITY_ORDER: List[int] = [1, 2, 3]

# Grouping keys for per-day processing
GROUP_KEYS: List[str] = ["person_id", "date"]



In [ ]:

def split_midnight_crossing_records(df: pd.DataFrame) -> pd.DataFrame:
    """Split any records that cross midnight into separate rows.

    The function preserves the original columns and adds/updates:
      - 'end_datetime': computed end timestamp
      - 'sleep_date'  : date on which the (sub)segment belongs

    Parameters
    ----------
    df : pd.DataFrame
        Requires columns: ['start_datetime', 'duration_in_min']

    Returns
    -------
    pd.DataFrame
        DataFrame with midnight-crossing rows split into two.
    """
    cur_df = df.copy()
    
    # Compute end time and midnight
    cur_df['end_datetime'] = pd.to_datetime(cur_df['start_datetime']) + \
                            pd.to_timedelta(cur_df['duration_in_min'], unit='m')
    midnight = pd.to_datetime(cur_df['start_datetime']).dt.floor('D') + pd.Timedelta(days=1)
    
    # Split records that cross midnight
    crosses_midnight = cur_df['end_datetime'] > midnight
    df_no_split = cur_df[~crosses_midnight].copy()
    df_no_split['sleep_date'] = pd.to_datetime(df_no_split['start_datetime']).dt.date
    df_split_before = cur_df[crosses_midnight].copy()
    df_split_after = cur_df[crosses_midnight].copy()

    # Calculate split durations
    df_split_before['duration_in_min'] = (
        (midnight[crosses_midnight] - pd.to_datetime(df_split_before['start_datetime']))
        .dt.total_seconds() / 60
    )
    df_split_after['duration_in_min'] = (
        (df_split_after['end_datetime'] - midnight[crosses_midnight])
        .dt.total_seconds() / 60
    )
    
    # Adjust dates and times
    df_split_before['sleep_date'] = pd.to_datetime(df_split_before['start_datetime']).dt.date
    df_split_after['sleep_date'] = midnight[crosses_midnight].dt.date
    df_split_after['start_datetime'] = midnight[crosses_midnight]

    return pd.concat([df_no_split, df_split_before, df_split_after], ignore_index=True)



# Overlap resolution 
def normalize_overlaps_sorted(rows):
    """
    rows: list of dicts with 'start','end','level','priority','is_main_sleep'
    If segments overlap, truncate the earlier segment to the next start.
    Drop segments that become non-positive length.
    """
    out = []
    for i, r in enumerate(rows):
        s, e = r['start'], r['end']
        if i + 1 < len(rows):
            ns = rows[i + 1]['start']
            if e > ns:
                e = ns
        if e > s:
            out.append({**r, 'end': e})
    return out

def subtract_covered(interval, schedule):
    """
    Return the parts of 'interval' not covered by 'schedule' (list of disjoint intervals).
    """
    s, e = interval
    if s >= e:
        return []
    remaining = [(s, e)]
    for (a, b) in schedule:
        next_remaining = []
        for (x, y) in remaining:
            if y <= a or x >= b:          # no overlap
                next_remaining.append((x, y))
            else:                          # cut out overlap
                if x < a:
                    next_remaining.append((x, a))
                if y > b:
                    next_remaining.append((b, y))
        remaining = [seg for seg in next_remaining if seg[1] > seg[0]]
        if not remaining:
            break
    return remaining

def add_to_schedule(intervals, schedule):
    """
    Insert non-overlapping 'intervals' into 'schedule' (disjoint list), filling only gaps.
    """
    added = []
    for (s, e) in intervals:
        gaps = subtract_covered((s, e), schedule)
        for (gs, ge) in gaps:
            added.append((gs, ge))
            schedule.append((gs, ge))
    schedule.sort(key=lambda x: x[0])
    return added

def build_day_schedule(df_day):
    """
    Process one (person_id, date) group:
      1) Main sleep first (priority 1 → 2 → 3)
      2) Non-main sleep fills remaining gaps (priority 1 → 2 → 3)
      3) Within each stream/priority, normalize overlaps by truncation
    """
    results = []
    schedule = []  # list[(start,end)] placed so far

    def process_stream(is_main_flag: bool):
        df_stream = df_day[df_day['is_main_sleep'] == is_main_flag]
        if df_stream.empty:
            return
        for p in PRIORITY_ORDER:
            sdf = df_stream[df_stream['priority'] == p].sort_values(['start', 'end'])
            if sdf.empty:
                continue
            rows = [{
                'start': r['start'],
                'end': r['end'],
                'level': r['level'],
                'priority': r['priority'],
                'is_main_sleep': r['is_main_sleep'],
            } for _, r in sdf.iterrows()]
            # truncate earlier-at-next-start within this stream+priority
            norm = normalize_overlaps_sorted(rows)
            for r in norm:
                # We always fill only gaps relative to what's already placed
                gaps = subtract_covered((r['start'], r['end']), schedule)
                if not gaps:
                    continue
                for (gs, ge) in gaps:
                    results.append({
                        'person_id': df_day['person_id'].iloc[0] if 'person_id' in df_day.columns else None,
                        'date': df_day['date'].iloc[0],
                        'start': gs,
                        'end': ge,
                        'duration_min': round((ge - gs).total_seconds() / 60, 3),
                        'level': r['level'],
                        'priority': r['priority'],
                        'is_main_sleep_source': is_main_flag,
                    })
                add_to_schedule(gaps, schedule)

    # 1) main sleep first
    process_stream(True)
    # 2) non-main only fills gaps
    process_stream(False)

    if not results:
        return pd.DataFrame(columns=[
            'person_id','date','start','end','duration_min','level','priority','is_main_sleep_source'
        ])
    return pd.DataFrame(results).sort_values('start').reset_index(drop=True)

def handle_overlapping_segments(
    day_data):
        
    """
    Input columns (as requested):
      - start_datetime, duration_in_min, level, is_main_sleep, person_id
    Output: non-overlapping segments with assigned level/priority that actually landed.
    """
    df = day_data.copy()
    # compute start/end, date, priority
    df['start_datetime'] = pd.to_datetime(df['start_datetime'])
    df['start'] = df['start_datetime']
    df['end'] = df['start_datetime'] + pd.to_timedelta(df['duration_in_min'].astype(float), unit='m')
    df['level'] = df['level'].astype(str)
    df['priority'] = df['level'].str.lower().map(PRIORITY_TABLE).fillna(4).astype(int)
    df['is_main_sleep'] = df['is_main_sleep'].astype(bool)
    df['person_id'] = df['person_id'] if 'person_id' in df.columns else None
    df['date'] = df['start'].dt.date

    # Only keep candidates with priority in {1,2,3}
    df = df[df['priority'].isin(PRIORITY_ORDER)]
    if df.empty:
        return pd.DataFrame(columns=[
            'person_id','date','start','end','duration_min','level','priority','is_main_sleep_source'
        ]).rename(columns = {'start':'start_datetime','duration_min':'duration_in_min',\
                                     'is_main_sleep_source':'is_main_sleep','end':'end_datetime'})

    outs = []
    for _, g in df.groupby(GROUP_KEYS, sort=False):
        outs.append(build_day_schedule(g))
    if not outs:
        return pd.DataFrame(columns=[
            'person_id','date','start','end','duration_min','level','priority','is_main_sleep_source'
        ]).rename(columns = {'start':'start_datetime','duration_min':'duration_in_min',\
                                     'is_main_sleep_source':'is_main_sleep','end':'end_datetime'})
    return pd.concat(outs, ignore_index=True).rename(columns = {'start':'start_datetime','duration_min':'duration_in_min',\
                                     'is_main_sleep_source':'is_main_sleep','end':'end_datetime'})


In [ ]:
# Day processor (stats)
def process_person_day(person_id, date, data):
    """Process one person's sleep for a single date.
    Returns
    -------
    tuple: (result_series, type_flag, total_duration, n_segments, minute_array, placed_segments_df)
    - type_flag is 'processed' or 'empty'
    """
    
    # Filter for specific date
    day_data = data[data['sleep_date'] == date]

    # Handle overlapping segments first
    merged_data = handle_overlapping_segments(day_data).drop_duplicates()

    # If total duration exceeds a day, mark as empty/problematic
    total_duration = merged_data['duration_in_min'].sum()
    if total_duration > 1440:

        main_dict_nan = {
            'person_id': person_id,
            'date': date,
        }
        for s in ['awake','restless','asleep','wake','light','rem','deep','unknown']:
            main_dict_nan[f'{s}_count'] = np.nan
            main_dict_nan[f'nonmain_{s}_count'] = np.nan

        main_dict_nan['unique_stages'] = np.nan
        main_dict_nan['nonmain_unique_stages'] = np.nan
        main_dict_nan['number_of_unique_stages'] = np.nan
        main_dict_nan['nonmain_number_of_unique_stages'] = np.nan
        
        
        return pd.Series(main_dict_nan), 'empty',total_duration, len(merged_data), \
            pd.DataFrame(columns=['person_id','start_datetime', 'end_datetime', 'level', 'is_main_sleep','duration_in_min'])

    
    # Counts per stream (main/nonmain)
    tru_m = merged_data[merged_data['is_main_sleep']==True]
    tru_uni = set(tru_m['level'])
    fal_m = merged_data[merged_data['is_main_sleep']==False]
    fal_uni = set(fal_m['level'])
    main_dict_true = {
        'person_id': person_id,
        'date': date,
    }

    # Counts of segments (not minutes) by label, per stream
    for s in ['awake','restless','asleep','wake','light','rem','deep','unknown']:
        main_dict_true[f'{s}_count'] = len(tru_m[tru_m['level']==s])
    for s in ['awake','restless','asleep','wake','light','rem','deep','unknown']:
        main_dict_true[f'nonmain_{s}_count'] = len(fal_m[fal_m['level']==s])
    main_dict_true['unique_stages'] = list(tru_uni) if len(list(tru_uni))>0 else np.nan
    main_dict_true['nonmain_unique_stages'] = list(fal_uni) if len(list(fal_uni))>0 else np.nan
    main_dict_true['number_of_unique_stages'] = len(list(tru_uni))
    main_dict_true['nonmain_number_of_unique_stages'] = len(list(fal_uni))
    
    merged_data['person_id'] = person_id
    return pd.Series(main_dict_true), 'processed',total_duration, len(merged_data), \
        merged_data[['person_id','start_datetime', 'end_datetime', 'level', 'is_main_sleep','duration_in_min']]

# Person processor: accumulates stats
def process_person(person_id, person_data, chunk_no):
    """Process one person's entire data (to be run in parallel).
     

    Returns a dictionary with:
    - results: per-day Series
    - stats: per-day minute statistics (time asleep, in bed, etc.)
    - full_pr_df: placed segments across all days
    - tot_len_entries / empty_days / valid_days / removed_days / any_valid_day
    """
    results = []
    dfs_min = []
    person_full_pr_df = []
    empty_d = 0
    proc_d = 0
    tot_len_entries = 0
    removed_days = {'person_id': [], 'date': [], 'total_duration': []}
    
    for date in person_data['sleep_date'].unique():
        
        result, type_, total_dur, len_entries, per_pr_df = process_person_day(
            person_id, date, person_data)

        


        if type_ == 'processed':
            proc_d+=1
            tot_len_entries += len_entries

            # Build per-day minute stats from the array
            ds = {}
            for jj in ['date','person_id','unique_stages','nonmain_unique_stages','number_of_unique_stages',\
                      'nonmain_number_of_unique_stages']:
                ds[jj] = result[jj]
            
        else:
            # Record removed/invalid day for logging
            empty_d += 1
            removed_days['person_id'].append(person_id)
            removed_days['date'].append(date)
            removed_days['total_duration'].append(total_dur)
        # Collect per-day outputs
        if result is not None:
            results.append(result)
            dfs_min.append(ds)
            person_full_pr_df.append(per_pr_df)
    
    # Return all results for this person
    return {
        'person_id': person_id,
        'results': pd.DataFrame(results) if results else pd.DataFrame(),
        'stats': pd.DataFrame(dfs_min) if results else pd.DataFrame(),
        'full_pr_df': pd.concat(person_full_pr_df, ignore_index=True) if person_full_pr_df else pd.DataFrame(),
        'tot_len_entries': tot_len_entries,
        'empty_days': empty_d,
        'valid_days': proc_d,
        'removed_days': removed_days,
        'any_valid_day': 1 if len(results)>0 else 0
    }



# Batch processor for a chunk    
def process_sleep_data(df: pd.DataFrame, chunk_no, n_processes) -> pd.DataFrame:
    """Top-level batch processor for a chunk of sleep data.


    Steps:
    1) Deduplicate (prefer main-sleep duplicates)
    2) Remove entries with duration > 1440 minutes
    3) Split midnight-crossing records
    4) Parallelize per-person processing by fixing overlapped entries


    Parameters
    ----------
    df : pd.DataFrame
    Input rows with at least: ['person_id','level','start_datetime','duration_in_min','is_main_sleep']
    chunk_no : int
    Chunk identifier for logging and output directory naming.
    n_processes : Optional[int]
    How many workers to use. If None, uses all available cores.


    Returns
    -------
    (
    combined_results,
    combined_stats,
    log_dict,
    removed_days_examples,
    removed_entries_examples,
    removed_duplicate_entries_count,
    combined_full_pr_df,
    )
    """

    
    
    log_dict = {'chunk_num': [], 'total_entries': [], 'total_days': [], 'total_participants': [],
           'entry_thresh_entries': [], 'entry_thresh_days': [], 'entry_thresh_participants': [],
           'after_split_entries': [], 'after_split_days': [],
           'total_duration_thresh_entries': [], 'total_duration_thresh_days': [], 'total_duration_thresh_participants': []}

    removed_entries_examples = {'person_id': [], 'start_datetime': [], 'level': [], 'duration_in_min': []}    
    removed_duplicate_entries_count = {'chunk_num': [], 'num_duplicate_entries_removed': []}


    # 1. Initial cleaning
    df = df.copy()
    df['sleep_date'] = df['start_datetime'].dt.date
    df['is_main_sleep'] = df['is_main_sleep'].apply(lambda x: True if x=='true' else (False if x=='false' else x))
    
    df['is_main_sleep'] = df['is_main_sleep'].astype(bool)
    
    # Step1: Handle duplicates keeping is_main_sleep=True
    df = df.sort_values('is_main_sleep', ascending=False).drop_duplicates(
        subset=['person_id', 'level', 'start_datetime', 'duration_in_min'],
        keep='first'
    )
    
    all_len = len(df)
    removed_duplicate_entries_count['chunk_num'].append(chunk_no)
    removed_duplicate_entries_count['num_duplicate_entries_removed'].append(all_len-len(df))
        
    log_dict['chunk_num'].append(chunk_no)
    log_dict['total_entries'].append(len(df))
    log_dict['total_days'].append(len(df.drop_duplicates(subset=['person_id','sleep_date'])))
    log_dict['total_participants'].append(len(df.drop_duplicates(subset=['person_id'])))
    
    
    
    # Step2: Remove entries > 1440 minutes
    removed_df = df[df['duration_in_min']>1440]
    removed_entries_examples['person_id'].extend(removed_df['person_id'].to_list())
    removed_entries_examples['start_datetime'].extend(removed_df['start_datetime'].to_list())
    removed_entries_examples['level'].extend(removed_df['level'].to_list())
    removed_entries_examples['duration_in_min'].extend(removed_df['duration_in_min'].to_list())
    
    df = df[df['duration_in_min'] <= 1440]
    
    log_dict['entry_thresh_entries'].append(len(df))
    log_dict['entry_thresh_days'].append(len(df.drop_duplicates(subset=['person_id','sleep_date'])))
    log_dict['entry_thresh_participants'].append(len(df.drop_duplicates(subset=['person_id'])))
    
    # 3) Split at midnight
    df = split_midnight_crossing_records(df)
    
    log_dict['after_split_entries'].append(len(df))
    log_dict['after_split_days'].append(len(df.drop_duplicates(subset=['person_id','sleep_date'])))
    
    
    
    # 4) Parallelize per-person processing
    person_ids = df['person_id'].unique()
    
    person_data_dict = {pid: df[df['person_id'] == pid] for pid in person_ids}
    
    # Define number of processes
    if n_processes is None:
        n_processes = multiprocessing.cpu_count()
    
    print(f"Processing {len(person_ids)} participants using {n_processes} cores")
    
    # Use joblib's Parallel and delayed for parallel processing
    results_pl = Parallel(n_jobs=n_processes)(
        delayed(process_person)(pid, person_data_dict[pid], chunk_no) 
        for pid in tqdm(person_ids, desc="Processing participants")
    )
    
    
    
    all_results = []
    all_stats = []
    all_full_pr_df = []
    total_entries = 0
    total_empty_days = 0
    total_processed_days = 0
    total_processed_people = 0
    removed_days_examples = {'person_id': [], 'date': [], 'total_duration': []}
    
    for result in results_pl:
        all_results.append(result['results'])
        all_stats.append(result['stats'])
        all_full_pr_df.append(result['full_pr_df'])
        total_entries += result['tot_len_entries']
        total_empty_days += result['empty_days']
        total_processed_days += result['valid_days']
        total_processed_people += result['any_valid_day']
        
        # Extend removed days info
        removed_days_examples['person_id'].extend(result['removed_days']['person_id'])
        removed_days_examples['date'].extend(result['removed_days']['date'])
        removed_days_examples['total_duration'].extend(result['removed_days']['total_duration'])
    
    # Combine all DataFrames
    combined_results = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
    combined_stats = pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()
    combined_full_pr_df = pd.concat(all_full_pr_df, ignore_index=True) if all_full_pr_df else pd.DataFrame()
    
    log_dict['total_duration_thresh_entries'].append(total_entries)
    log_dict['total_duration_thresh_days'].append(total_processed_days)
    log_dict['total_duration_thresh_participants'].append(total_processed_people)
    

    
    
    
    
    
    return (
        combined_results, 
        combined_stats, 
        log_dict,
        removed_days_examples, 
        removed_entries_examples, 
        removed_duplicate_entries_count, 
        combined_full_pr_df
    )

    


### QCing

In [ ]:
folder_sleep = paths_dict.get("sleep_import")
for n_chunk in range(1,38):#adjust depending on whether processing ehr or no_ehr data
    
    sleep_df_s = pd.read_csv(f'{bucket_}/{folder_ehr_save}/chunk_{no}/fitbit_sleep_data_chunk_{n_chunk}.tsv', sep='\t')
    sleep_df_s['start_datetime'] = pd.to_datetime(sleep_df_s['start_datetime'])

    #run code for each chunk
    processed_df_s, dfs_min_df, log_s, r_days_s, r_entries_s, r_dup_s, ovelp_df_s = process_sleep_data(sleep_df_s,n_chunk, 32)



    #save dfs
    ovelp_df_s.to_csv(f'{bucket_}/{folder_sleep}/overlapped_entries_df_chunk_{n_chunk}.tsv',sep='\t',index = False)

    log_df = pd.DataFrame.from_dict(log_s)
    log_df.to_csv(f'{bucket_}/{folder_sleep}/log_for_chunk_{n_chunk}.tsv',sep='\t',index = False)
    removed_days_examples_df = pd.DataFrame.from_dict(r_days_s)
    removed_days_examples_df.to_csv(f'{bucket_}/{folder_sleep}/removed_days_examples_for_{n_chunk}.tsv',sep='\t',index = False)
    removed_entries_examples_df = pd.DataFrame.from_dict(r_entries_s)
    removed_entries_examples_df.to_csv(f'{bucket_}/{folder_sleep}/removed_entries_examples_for_{n_chunk}.tsv',sep='\t',index = False)
    removed_duplicate_entries_count_df = pd.DataFrame.from_dict(r_dup_s)
    removed_duplicate_entries_count_df.to_csv(f'{bucket_}/{folder_sleep}/removed_duplicate_entries_counts_for_{n_chunk}.tsv',sep='\t',index = False)

    dfs_min_df.to_csv(f'{bucket_}/{folder_sleep}/sleep_stats_df_chunk_{n_chunk}.tsv',sep='\t',index = False)

    

# Some extra dataframes used in later part of analysis

## followup estimation

this query grabs daily level hr data availability based on dates so we can estimate followup patterns

In [ ]:
bucket_ = os.environ.get('WORKSPACE_BUCKET')

def load_person_ids_from_file(file_path: str) -> list:
    """Load person IDs from a text file."""
    fs = gcsfs.GCSFileSystem()
    with fs.open(file_path, 'r') as file:
        person_ids = [int(line.strip()) for line in file.readlines()]
    return person_ids
no = 1 #change depending on chunk
pids_to_write = load_person_ids_from_file(f'{bucket_}/{folder_ehr_save}/chunk_{no}.txt')
set_no = 'first'
#can process maybe 5 chunks at once since we dont pull array level data here
for j in range(1,5):
    pids_to_write = pids_to_write + load_person_ids_from_file(f'{bucket_}/{folder_ehr_save}/chunk_{no+j}.txt')

    
person_ids_str = ', '.join(map(str, pids_to_write))

st = time.time()
dataset_97932049_hr_mapped_level_sql = """
SELECT
    person_id,
        CAST(datetime AS DATE) as date,
FROM
    `""" + os.environ["WORKSPACE_CDR"] + """.heart_rate_minute_level`
WHERE
    person_id IN (""" + person_ids_str + """)
GROUP BY
    person_id,
    date
"""


dataset_97932049_hr_mapped_level_df3 = pd.read_gbq(
    dataset_97932049_hr_mapped_level_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

end = time.time()
print(end-st)
df_s = dataset_97932049_hr_mapped_level_df3.copy()
dfs = df_s.groupby('person_id')['date'].apply(list).reset_index()
dfs = dfs.explode('date')
dfs['date'] = pd.to_datetime(dfs['date'])
dfs['year_month'] = dfs['date'].dt.to_period('M').astype(str)

# Pivot table to count occurrences
resultdfs = dfs.pivot_table(index='person_id', columns='year_month', aggfunc='size', fill_value=0)

resultdfs = resultdfs.reset_index()
resultdfs.to_csv(f'{set_no}_5000.tsv',sep='\t')

In [ ]:
##aggregate all sets in one file
dfs =[]
for j in ['first','second','third','fourth','fifth','sixth','seventh','eighth']:
    dfs.append(pd.read_csv(f'{j}_5000.tsv',sep='\t'))
    
df_all = pd.concat(dfs,ignore_index = True)
df_all = df_all.drop(columns = ['Unnamed: 0'])
df_all = df_all.fillna(0)
df_all = df_all[['person_id']+sorted([i for i in df_all.columns.tolist()[1:]])]
df_all = df_all.astype(int)
df_all.to_csv(f'{bucket_}/{folder_ehr_save}/{paths_dict.get("dates_hr_file_ehr")}.tsv',sep='\t',index = False)

Repeat same for people without ehr

In [ ]:
df_all1 = pd.read_csv(f'{bucket_}/{folder_ehr_save}/{paths_dict.get("dates_hr_file")}.tsv',sep='\t')
df_all2 = pd.read_csv(f'{bucket_}/{folder_no_ehr_save}/{paths_dict.get("dates_hr_file")}.tsv',sep='\t')
df_all = pd.concat([df_all1,df_all2],ignore_index = True)

date_cols = [col for col in df_all.columns if col != 'person_id']
df_long = df_all.melt(id_vars=['person_id'], var_name='year_month', value_name='days')
df_long['year_month'] = pd.to_datetime(df_long['year_month'], format='%Y-%m')
df_long
# Identify first and last available data point for each person
df_summary = df_long[df_long['days'] > 0].groupby('person_id').agg(
    first_active=('year_month', 'min'),
    last_active=('year_month', 'max')
).reset_index()

# Get list of months where person has data
df_active_months = df_long[df_long['days'] > 0].groupby('person_id')['year_month'].apply(set).reset_index()

# Merge summaries
df_summary = df_summary.merge(df_active_months, on='person_id')


# Count the number of months each person has data
df_counts = df_long[df_long['days'] > 0].groupby('person_id').agg(
    total_months=('year_month', 'count')
).reset_index()

# Merge both summaries
df_summary = df_summary.merge(df_counts, on='person_id')


# Generate full expected date range for each person
def find_missing_months(row):
    full_range = pd.date_range(start=row['first_active'], end=row['last_active'], freq='MS')
    observed_months = row['year_month']  # Set of months where data exists
    missing = [d.strftime('%Y-%m') for d in full_range if d not in observed_months]
    return missing if missing else None  # Return missing months or None
df_summary = df_summary
df_summary['missing_months'] = df_summary.apply(find_missing_months, axis=1)

# # Count number of missing months
df_summary['missing_count'] = df_summary['missing_months'].apply(lambda x: len(x) if x else 0)
def check_continuity_by_year_old(row):
    if not row['missing_months']:
        return 'Continuous'
    
    # Convert missing months to datetime for year grouping
    missing_dates = pd.to_datetime([f"{m}-01" for m in row['missing_months']])
    
    # Count missing months by year
    missing_by_year = missing_dates.year.value_counts()
    
    if missing_by_year.max() > 0:
        return 'Interrupted'
    else:
        return 'Continuous'

df_summary['data_continuity'] = df_summary.apply(check_continuity_by_year_old, axis=1)


In [ ]:
# Function to find longest "relaxed" streak (allowing ≤2 missing months per year)
def find_longest_relaxed_streak(row):
    # For continuous data, return the full range
    if row['data_continuity'] == 'Continuous':
        return {
            'longest_streak': row['total_months'],
            'streak_start': row['first_active'],
            'streak_end': row['last_active']
        }
    
    # Get all months with data as a sorted list
    active_months = sorted(list(row['year_month']))
    if not active_months:
        return {'longest_streak': 0, 'streak_start': None, 'streak_end': None}
    
    # Find gaps between months
    gaps = []
    for i in range(1, len(active_months)):
        prev = active_months[i-1]
        curr = active_months[i]
        
        # Calculate month difference
        month_diff = (curr.year - prev.year) * 12 + curr.month - prev.month
        
        if month_diff > 1:
            # Check if  there's a gap (≤ 3 months)
            if month_diff <= 3:
                # Check if the gap crosses a year boundary
                if prev.year != curr.year:
                    months_in_prev_year = 12 - prev.month
                    months_in_curr_year = curr.month - 1
                    if months_in_prev_year > 2 or months_in_curr_year > 2:
                        gaps.append(i)
                else:
                    # Gap within same year - if > 2 months, it's a real gap
                    if month_diff > 3:
                        gaps.append(i)
            else:
                gaps.append(i)
    
    # Add start and end indices to calculate segments
    all_breaks = [0] + gaps + [len(active_months)]
    
    # Find the longest segment
    max_length = 0
    start_idx = end_idx = 0
    
    for i in range(len(all_breaks)-1):
        segment_length = all_breaks[i+1] - all_breaks[i]
        if segment_length > max_length:
            max_length = segment_length
            start_idx = all_breaks[i]
            end_idx = all_breaks[i+1] - 1
    
    return {
        'longest_streak': max_length,
        'streak_start': active_months[start_idx],
        'streak_end': active_months[end_idx]
    }

# Apply the function to get longest streak with relaxed gap rules
streak_info = df_summary.apply(find_longest_relaxed_streak, axis=1)

# Extract fields from results
df_summary['longest_relaxed_streak'] = streak_info.apply(lambda x: x['longest_streak'])
df_summary['longest_streak_start'] = streak_info.apply(lambda x: x['streak_start'])
df_summary['longest_streak_end'] = streak_info.apply(lambda x: x['streak_end'])

In [ ]:
df_summary['first_active_year'] = df_summary['first_active'].apply(lambda x: int(str(x)[0:4]))
df_summary['first_active_month'] = df_summary['first_active'].apply(lambda x: int(str(x)[5:7]))
df_summary['last_active_year'] = df_summary['last_active'].apply(lambda x: int(str(x)[0:4]))
df_summary['last_active_month'] = df_summary['last_active'].apply(lambda x: int(str(x)[5:7]))
df_summary['expected_months'] = df_summary.apply(lambda x: \
(x['last_active'].year - x['first_active'].year) * 12 + x['last_active'].month - x['first_active'].month+1, axis = 1)
df_summary.to_csv(f'{bucket_}/{paths_dict.get("followup_heartrate_file")}.tsv',sep='\t',index = False)

## Person level demographic and other details

In [ ]:
# This query represents dataset "fitbit_all" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_13847333_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_fitbit = 1 ) )"""

df_person = pd.read_gbq(
    dataset_13847333_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")


In [ ]:
import pandas
import os

# This query represents dataset "fitbit_all" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_13847333_no_consent_sql = """
    SELECT
        person.person_id    
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_fitbit = 1 AND has_wear_consent = 0) )"""

dataset_13847333_no_consent_df = pd.read_gbq(
    dataset_13847333_no_consent_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")
dataset_13847333_no_consent_df['wear_consent'] = 0

dataset_13847333_consent_sql = """
    SELECT
        person.person_id    
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_fitbit = 1 AND has_wear_consent = 1) )"""

dataset_13847333_consent_df = pd.read_gbq(
    dataset_13847333_consent_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")
dataset_13847333_consent_df['wear_consent'] = 1


dataset_13847333_no_ehr_sql = """
    SELECT
        person.person_id    
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_fitbit = 1 AND has_ehr_data = 0) )"""

dataset_13847333_no_ehr_df = pd.read_gbq(
    dataset_13847333_no_ehr_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")
dataset_13847333_no_ehr_df['ehr'] = 0

dataset_13847333_ehr_sql = """
    SELECT
        person.person_id    
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_fitbit = 1 AND has_ehr_data = 1) )"""

dataset_13847333_ehr_df = pd.read_gbq(
    dataset_13847333_ehr_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")
dataset_13847333_ehr_df['ehr'] = 1



dataset_13847333_age_sql = """
    SELECT
        * 
    FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_fitbit = 1 """

df_age = pd.read_gbq(
    dataset_13847333_age_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")



consent_df = pd.concat([dataset_13847333_consent_df,dataset_13847333_no_consent_df],ignore_index = True)
ehr_df = pd.concat([dataset_13847333_ehr_df,dataset_13847333_no_ehr_df],ignore_index = True)

df_person = df_person.merge(consent_df,on = 'person_id')
df_person = df_person.merge(ehr_df,on = 'person_id')

df_person = df_person.merge(df_age[['person_id','age_at_cdr']],on = 'person_id')

df_person['Race'] = df_person['race'].apply(lambda x: 'Other' if x \
                    not in['White', 'Black or African American', 'Asian'] else x)


In [ ]:
# This query represents dataset "fitbit_all" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_90262036_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (1585857, 1585952, 1585375, 1585940, 1586198, 1586201, 1586207)
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_fitbit = 1 ) )
        )"""

df_survey = pd.read_gbq(
    dataset_90262036_survey_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")


In [ ]:
df_survey['category'] = df_survey['question'].apply(lambda x: 'Education' if 'education' in x.lower() else 'other')
df_cat = df_survey[df_survey['category'] == 'Education']
df_cat['answer'].value_counts()
df_cat['education_level'] = df_cat['answer'].apply(lambda x: \
            'Undefined' if x in ['PMI: Skip','PMI: Prefer Not To Answer'] else \
             ('College Degree or Advanced' if x in ['Highest Grade: Advanced Degree',\
               'Highest Grade: College Graduate'] else \
              ('Some college degree' if x in ['Highest Grade: College One to Three'] else \
               'No college degree')))



df_survey['employment'] = df_survey['question'].apply(lambda x: 'Employment' if 'employment' in x.lower() else 'other')
df_cat_e = df_survey[df_survey['employment'] == 'Employment']
df_cat_e['answer'].value_counts()
df_cat_e['employment_type'] = df_cat_e['answer'].apply(lambda x: \
            'Unknown' if x in ['PMI: Skip','PMI: Prefer Not To Answer'] else \
             ('Retired' if x in ['Employment Status: Retired'] else \
('Not working' if x in ['Employment Status: Out Of Work One Or More',\
                        'Employment Status: Out Of Work Less Than One',\
                       'Employment Status: Unable To Work'] else \
               ('Homemaker' if x in ['Employment Status: Homemaker'] else \
         ('Student' if x in ['Employment Status: Student'] else \
          ('Job' if x in ['Employment Status: Employed For Wages'] else 'Self Employed'))))))


df_survey['income'] = df_survey['question'].apply(lambda x: 'Income' if 'income' in x.lower() else 'other')
df_cat_i = df_survey[df_survey['income'] == 'Income']
df_cat_i['answer'].value_counts()
df_cat_i['income_level'] = df_cat_i['answer'].apply(lambda x: \
            'Unknown' if x in ['PMI: Skip','PMI: Prefer Not To Answer'] else \
             ('100K-200K' if x in ['Annual Income: 100k 150k','Annual Income: 150k 200k'] else \
              ('>200K' if x in ['Annual Income: more 200k'] else \
               ('<10K' if x in ['Annual Income: less 10k'] else \
         ('10K-50K' if x in ['Annual Income: 10k 25k','Annual Income: 25k 35k','Annual Income: 35k 50k'] else \
          '50K-100K')))))

In [ ]:
df_summary = pd.read_csv(f'{bucket_}/{paths_dict.get("followup_heartrate_file")}.tsv',sep='\t')

df_summary['first_active'] = pd.to_datetime(df_summary['first_active'])
df_cat_e = df_cat_e.merge(df_summary[['person_id','first_active']],on = 'person_id')
df_cat_e['survey_datetime'] = df_cat_e['survey_datetime'].apply(lambda x: str(x)[:10])
df_cat_e['survey_datetime'] = pd.to_datetime(df_cat_e['survey_datetime'])

df_cat_e['date_diff'] = (df_cat_e['survey_datetime'] - df_cat_e['first_active']).abs()
df_cat_e = df_cat_e.sort_values('date_diff').drop_duplicates('person_id', keep='first')
df_cat_e = df_cat_e.drop(columns='date_diff')

In [ ]:
df_person = df_person.merge(df_cat[['person_id','education_level']],on = 'person_id')
df_person = df_person.merge(df_cat_i[['person_id','income_level']],on = 'person_id')
df_person = df_person.merge(df_cat_e[['person_id','employment_type']],on = 'person_id',how = 'outer')
df_person['employment_type'] = df_person['employment_type'].apply(lambda x: \
                                                                 x if not pd.isna(x) else 'Unknown')


In [ ]:
df_person.to_csv(f'{bucket_}/{paths_dict.get("person_stats_file")}.tsv',sep='\t',index = False)

## Zip code data

In [ ]:
#This query pulls zipcode data and other social determinants of health
dataset_zip_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3
    WHERE
        observation.PERSON_ID IN (SELECT
            distinct person_id
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p
            WHERE
                has_fitbit = 1 )
            
             )
        AND observation_source_concept_id = 1585250
        AND observation.value_as_string NOT LIKE 'Res%'"""

df_zip = pd.read_gbq(
    dataset_zip_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")
df_zip.to_csv(f'{bucket_}/{paths_dict.get("zip_code_file")}.tsv', sep='\t',index = False)

## Person-level activity minutes summary stats reported by fitbit

In [ ]:
dfs_avg_activity = []
for no in range(1,3):
    if no<38:
        activity_summary_df = pd.read_csv(f'{bucket_}/{folder_ehr_save}/chunk_{no}/fitbit_activity_summary_chunk_{no}.tsv',sep='\t')
    else:
        activity_summary_df = pd.read_csv(f'{bucket_}/{folder_no_ehr_save}/chunk_{no}/fitbit_activity_summary_chunk_{no}.tsv',sep='\t')
    activity_summary_df['moderate_to_vigorous_minutes'] = activity_summary_df['fairly_active_minutes']+ activity_summary_df['very_active_minutes']
    activity_cols = ['activity_calories','calories_bmr','calories_out','elevation','fairly_active_minutes','floors',\
    'lightly_active_minutes','marginal_calories','sedentary_minutes','steps','very_active_minutes','moderate_to_vigorous_minutes']
    avg_activity = activity_summary_df.groupby('person_id')[activity_cols].mean().reset_index()
    avg_activity.columns = ['person_id'] + [f'avg_{col}' for col in activity_cols]
    dfs_avg_activity.append(avg_activity)

Avg_activity = pd.concat(dfs_avg_activity,ignore_index = True)    
Avg_activity.to_csv(f'{bucket_}/{paths_dict.get("person_activity_summ_file")}.tsv',sep='\t',index = False)


# Follow along in the 'release_02_v8_qc_ingestion' notebook for next steps
There we do quality control and ingest data from csv files into set of HD5 files per person